# Déploiement des modèles sur SpiNNaker (SANDS)

Ce notebook lance `spinnaker_faithful_eval.py` dans le noyau Jupyter sPyNNaker, où `pyNN.spiNNaker` est installé, puis calcule l'énergie physique de chaque run.

Il s'exécute depuis la racine du dépôt copiée dans `work/` (voir `SpiNNaker/README.md`), cellules dans l'ordre. Tous les résultats sont écrits dans `results_spinnaker/`.

## 1. Dossier de travail et dépendances

In [ ]:
import csv
import importlib.util
import json
import os
import site
import subprocess
import sys

if not os.path.exists("spinnaker_plan.json") and os.path.exists(
    "../spinnaker_plan.json"
):
    os.chdir("..")
assert os.path.exists("spinnaker_plan.json"), (
    "spinnaker_plan.json introuvable : ouvrir le notebook depuis la racine du dépôt"
)
print("Dossier de travail :", os.getcwd())

RESULTS = "results_spinnaker"
DATASETS = ["nmnist", "smnist", "shd", "ssc"]


def pip_install(*packages):
    """Installe dans l'environnement du noyau (et non dans celui du terminal)."""
    command = [sys.executable, "-m", "pip", "install", "--quiet", *packages]
    if subprocess.call(command) != 0:
        subprocess.check_call(command[:4] + ["--user"] + command[4:])
    importlib.invalidate_caches()
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())


try:
    import torch
except ImportError:
    pip_install("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
try:
    import spikingjelly  # noqa: F401
except ImportError:
    pip_install("--no-deps", "spikingjelly")

## 2. Vérification de l'environnement

`énergie` doit valoir `True` : sinon sPyNNaker n'écrit pas de rapport d'énergie (le script crée `spynnaker.cfg` s'il n'existe pas).

In [ ]:
import configparser

import pyNN.spiNNaker as sim
import torch

sys.path.insert(0, os.path.join(os.getcwd(), "SpiNNaker"))
import spinnaker_energy
import spinnaker_faithful_eval as faithful

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("spikingjelly:", importlib.util.find_spec("spikingjelly") is not None)
print("pyNN        :", sim.__name__)
print("neurone     :", faithful.delta_neuron_model(sim).__name__)

config = configparser.ConfigParser(strict=False)
config.read([os.path.expanduser("~/.spynnaker.cfg"), "spynnaker.cfg"])
print("énergie     :", config.get("Reports", "write_energy_report", fallback="False"))

## 3. Émulation sans la carte (quelques minutes)

Chaque run est converti et émulé en virgule fixe. Pour chacun : bits d'entrée, facteur de ralentissement minimal (pour que les cœurs traitent leurs spikes à temps **et** que le réseau tienne sur la carte), coût estimé du cœur le plus chargé, nombre de cœurs.

La cellule en déduit `FACTEUR`, le facteur commun à tous les runs (le plus grand nécessaire), et `PIRE`, le run le plus chargé. Seuls les jeux de `DATASETS` tiennent sur une carte SpiNN-5 ; les autres sont refusés (`TOO_LARGE`).

In [ ]:
for dataset in DATASETS:
    faithful.main(["--dataset", dataset, "--emulate", "--time-scale-factor", "auto"])

with open(f"{RESULTS}/summary_emulation.csv", newline="", encoding="utf-8") as f:
    emulation = {r["run_id"]: r for r in csv.DictReader(f)}
ok = {k: r for k, r in emulation.items() if r["status"] == "EMULATED"}
for r in ok.values():
    print(
        f"{r['run_id']:<38} bits={r['input_bits']:>2} "
        f"facteur={r['needed_time_scale_factor']:>4} coût={r['core_cost_ms']:>7} ms "
        f"spikes/cœur={r['peak_core_spikes']:>6} cœurs={r['n_cores']:>5}"
    )
for k, r in emulation.items():
    if r["status"] != "EMULATED":
        print(f"{k:<38} {r['status']} : {r['error_msg']}")

FACTEUR = str(max(int(r["needed_time_scale_factor"]) for r in ok.values()))
PIRE = max(
    ok.values(),
    key=lambda r: (int(r["needed_time_scale_factor"]), float(r["core_cost_ms"])),
)
print(
    f"\n{len(ok)}/{len(emulation)} runs déployables | facteur commun : {FACTEUR} | "
    f"run le plus chargé : {PIRE['run_id']}"
)

## 4. Test du run le plus chargé (quelques minutes)

Attendu : `fiable=True`, `tardifs=0` et `accord` ≥ 90 %. Sinon, ne pas lancer la section 5 : la colonne `late_by_population` indique les couches qui reçoivent trop de spikes, et il faut augmenter `FACTEUR`.

In [ ]:
plan = json.load(open("spinnaker_plan.json", encoding="utf-8"))
with open("plan_test.json", "w", encoding="utf-8") as f:
    json.dump([e for e in plan if faithful.tee.run_id_of(e) == PIRE["run_id"]], f)
faithful.main(
    [
        "--plan",
        "plan_test.json",
        "--results-dir",
        f"{RESULTS}_test",
        "--time-scale-factor",
        FACTEUR,
    ]
)

with open(f"{RESULTS}_test/summary_benchmark.csv", newline="", encoding="utf-8") as f:
    test = list(csv.DictReader(f))[-1]
print("tardifs par population :", test["late_by_population"] or "aucun")

## 5. Tous les modèles sur la carte

Même facteur pour tous les runs (`FACTEUR`, section 3) : l'énergie physique est proportionnelle à la durée d'exécution, donc au facteur, et doit rester comparable d'un run à l'autre. Les résultats de l'article ont été obtenus avec un facteur de 30.

Durée : environ 3 h + 0,2 h × `FACTEUR` / 10. La cellule peut être relancée : les runs déjà réussis sont sautés.

In [ ]:
print("Facteur utilisé :", FACTEUR)
for dataset in DATASETS:
    faithful.main(["--dataset", dataset, "--time-scale-factor", FACTEUR])

## 6. Précision et fiabilité

Un run est exploitable si `fiable` vaut `True` (aucun paquet perdu, aucun spike tardif, aucun dépassement du timer) et si `accord` (réponses identiques à l'émulation en virgule fixe) est d'au moins 90 %.

In [ ]:
with open(f"{RESULTS}/summary_benchmark.csv", newline="", encoding="utf-8") as f:
    rows = {r["run_id"]: r for r in csv.DictReader(f)}

print(
    f"{'run':<38} {'pytorch':>7} {'v.fixe':>7} {'carte':>6} {'accord':>6} "
    f"{'n/cœur':>6} {'perdus':>7} {'tardifs':>7} {'retards':>7} {'fiable':>6}"
)
a_refaire = []
for row in rows.values():
    print(
        f"{row['run_id']:<38} {row['accuracy_pytorch_pct']:>7} "
        f"{row['accuracy_fixed_point_pct']:>7} {row['accuracy_pct']:>6} "
        f"{row['agreement_fixed_point_pct']:>6} {row['min_neurons_per_core']:>6} "
        f"{row['lost_input_packets']:>7} {row['late_spikes']:>7} "
        f"{row['timer_overruns']:>7} {row['reliable']:>6}"
    )
    if row["status"] != "SUCCESS" or row["reliable"] != "True":
        a_refaire.append(row["run_id"])
print(
    f"\n{len(rows) - len(a_refaire)}/{len(rows)} runs fiables. À refaire : {a_refaire}"
)

## 7. Énergie physique et tableau final

Écrit `results_spinnaker/spinnaker_energy.csv` (une ligne par run réussi) et, si `theoretical_energy_results.csv` est présent à la racine, `results_spinnaker/final_energy_results.csv` (énergie théorique et énergie physique côte à côte).

In [ ]:
spinnaker_energy.main(["--results-dir", RESULTS])